In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [2]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index

In [4]:
from openai import OpenAI


In [3]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [5]:
documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

In [6]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [8]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course, the content of the course, or the logistics of the course.
The oftopic questions shouldn't be answered. If you can't find the answer in the search results, say you don't know.    

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [11]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [12]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [17]:
result = runner.loop(
    prompt="what's queen gambit?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [15]:
result.cost

CostInfo(input_cost=Decimal('0.000735'), output_cost=Decimal('0.0005085'), total_cost=Decimal('0.0012435'))

In [16]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nThe question has to be about the course, the content of the course, or the logistics of the course.\nThe oftopic questions shouldn't be answered. If you can't find the answer in the search results, say you don't know.    \n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content="what's queen gambit?", role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"queen gambit course chess opening Queen\'s Gambit"}', call_id='call_rA2mGpMz2U8SQ7InPO9YvFHW', name='sear